# Chapter 8 — 네트워크/웹 로그 데이터의 이해 (실습)

## 실습 목표
1. CICIDS2017 합성 부분집합의 열·자료형·결측치 진단
2. 라벨별 통계 비교 (groupby)
3. pivot_table 로 라벨 × 포트 매트릭스
4. 언더샘플링·오버샘플링 분포 비교

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def make_rows(n, label, dur, pkts, syn, ports):
    return pd.DataFrame({
        'Source IP': rng.choice([f'10.0.0.{i}' for i in range(1,50)], size=n),
        'Destination Port': rng.choice(ports, size=n),
        'Flow Duration': rng.uniform(*dur, size=n),
        'Total Fwd Packets': rng.integers(*pkts, size=n),
        'SYN Flag Count': rng.uniform(*syn, size=n),
        'Protocol': rng.choice([6, 17], size=n, p=[0.85, 0.15]),  # 6=TCP, 17=UDP
        'Label': label,
    })

df = pd.concat([
    make_rows(16000, 'BENIGN',   (1000, 1_000_000), (5, 100),    (0, 0.05),  [80, 443, 53, 22]),
    make_rows( 2300, 'DDoS',     (10, 5000),         (500, 5000), (0.4, 0.7), [80, 443]),
    make_rows( 1200, 'PortScan', (1, 200),           (1, 5),      (0.85, 1.0), list(range(1, 1024))),
], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)

# inf 가 있는 컬럼 시뮬레이션 (CICIDS2017 의 Flow Bytes/s 같은 컬럼)
df['Flow Bytes/s'] = np.where(df['Flow Duration'] < 0.001, np.inf, 1000 / df['Flow Duration'].clip(lower=1e-6))
print(f'shape: {df.shape}')
df.head()

---
## Step 1. 진단 — 컬럼·dtype·결측·inf

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 진단 — 컬럼·dtype·결측·inf
# 만들 결과: inf_cols, n_inf
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
...

inf_cols = ...
...
n_inf = ...
...
```
</details>


In [ ]:
# inf → NaN 변환
df = df.replace([np.inf, -np.inf], np.nan)
print(f'변환 후 NaN: {df.isna().sum().sum()}')

---
## Step 2. 라벨별 통계

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 라벨별 통계
# 만들 결과: stats, n_flows, mean_pkts, mean_dur, mean_syn
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
stats = ...
    n_flows = ...
    mean_pkts = ...
    mean_dur = ...
    mean_syn = ...
...
```
</details>


---
## Step 3. pivot_table — 라벨 × 포트

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. pivot_table — 라벨 × 포트
# 만들 결과: top_ports, pt, index, aggfunc
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
top_ports = ...
pt = ...
    index = ...
    aggfunc = ...
...
```
</details>


---
## Step 4. 언더샘플링 vs 오버샘플링

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. 언더샘플링 vs 오버샘플링
# 만들 결과: min_count, under, max_count, over, counts
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 언더샘플링 — 다수 클래스를 최소 클래스 수에 맞춤
min_count = ...
under = ...
    g.sample(n = ...
]).reset_index(drop = ...

# 오버샘플링 — 소수 클래스를 다수 클래스 수에 맞춤 (단순 복원추출)
max_count = ...
over = ...
    g.sample(n = ...
]).reset_index(drop = ...

fig, axes = ...
for ax, (title, d) in zip(axes, [('Original', df), ('Undersampled', under), ('Oversampled', over)]):
    counts = ...
    ax.bar(counts.index, counts.values, color = ...
    ax.set_title(f'{title} (n = ...
    ax.tick_params(axis = ...
...
```
</details>


---
## 정리

- 보안 로그는 결측·inf·불균형이 거의 항상 존재 — 사전 진단이 필수.
- pivot_table 로 라벨 × 다른 차원의 분포를 한 번에 본다.
- 언더/오버샘플링은 단순하지만 매우 효과적인 첫 대응책.
- ch09 에서 본격 전처리(결측·이상·정규화·인코딩)로 들어간다.